# 第 6 章 · Planning（CrewAI）

书里的做法：把"先列计划、再按计划写"写进**一条 task 的两句指令**里，由同一个 agent 完成。

⚠️ `process=Process.sequential` 只决定"任务之间怎么排"，**它不做规划**。
规划藏在 `Task.description` 那两句话里——计划是**散文**，不是数据。

所以下面把它改造成**可校验的结构**：要求模型交 JSON，再用 `ArticlePlan` 校验，
`plan` 变成真正的 `list[str]`。同一个任务、同一个目标，只是计划从"一段文字"变成了"一张表"。

kernel 选 **`Python (agentic-crewai)`**。


In [1]:
import os
import json
import re
import threading

from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, ValidationError

# 【适配】CrewAI 1.15 改用原生 provider：Agent.llm 只接受 str | BaseLLM，不再收 LangChain 对象，
#   且 "openai/xxx" 前缀要走 litellm 回落（未安装）。deepseek 是内置 provider，用 deepseek/ 前缀。
llm = LLM(model="deepseek/deepseek-flash",
          api_key=os.environ["OPENAI_API_KEY"],
          base_url="https://api.deepseek.com/v1")


# ★ 本章新增：把"计划"描述成一张表 —— 一个栏位放提纲（多条），一个栏位放正文。
#   这个类不执行任何东西，只是声明"要按什么结构收作业"。
class ArticlePlan(BaseModel):
    plan: list[str]
    summary: str


# 【适配】CrewAI 的 output_pydantic / output_json 依赖 OpenAI 的 response_format 结构化接口，
#   DeepSeek 直接返回 400（"This response_format type is unavailable now"）。
#   所以约束改由我们自己做：要求模型输出 JSON，再用 ArticlePlan 校验。
def parse_plan(text: str) -> ArticlePlan:
    """从模型输出里抠出 JSON 并按 ArticlePlan 校验。不合法就抛 ValidationError。"""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    return ArticlePlan.model_validate_json(cleaned)


planner_writer_agent = Agent(
    role='Article Planner and Writer',
    goal='Plan and then write a concise, engaging summary on a specified topic.',
    backstory=(
        'You are an expert technical writer and content strategist. '
        'Your strength lies in creating a clear, actionable plan before writing, '
        'ensuring the final summary is both informative and easy to digest.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

topic = "The importance of Reinforcement Learning in AI"
high_level_task = Task(
    description=(
        f"1. Create a bullet-point plan for a summary on the topic: '{topic}'.\n"
        f"2. Write the summary based on your plan, keeping it around 200 words."
    ),
    # 交付形态从"两段 markdown"改成"一个 JSON 对象" —— 这样才可能被程序取用
    expected_output=(
        'A JSON object with exactly two keys:\n'
        '  "plan": a list of short strings (the bullet points);\n'
        '  "summary": a string (the ~200 word summary).\n'
        'Output ONLY the JSON, no markdown fences.'
    ),
    agent=planner_writer_agent,
)

# ★ 本章新增 Process：决定"任务之间怎么排"。sequential = 挨个跑，不是并行、也不是规划。
crew = Crew(
    agents=[planner_writer_agent],
    tasks=[high_level_task],
    process=Process.sequential,
)


# 【适配】kernel 里有运行中的事件循环，同步 kickoff() 会抛 RuntimeError → 放进独立线程（同第 5 章）
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


print("## Running the planning and writing task ##")
result = run_in_thread(crew.kickoff)
raw = result.raw

print("\n---\n## 校验结果 ##\n---")
try:
    plan_obj = parse_plan(raw)
except ValidationError as e:
    print("❌ 校验失败 —— 模型没按表格交作业：")
    print(e)
    print("\n--- 原始文本 ---")
    print(raw)
else:
    # 到这里，plan / summary 一定是"列表 / 字符串"且栏位齐全
    print("✅ 通过校验，拿到的是:", type(plan_obj).__name__)
    print(f"\n计划共 {len(plan_obj.plan)} 条（这是 list，可按下标取）：")
    for i, item in enumerate(plan_obj.plan, 1):
        print(f"  {i}. {item}")
    print("\n正文（字符串，前 200 字符）：")
    print(plan_obj.summary[:200])
    print("\n---\n## 原始文本（未加工）##\n---")
    print(raw)


## Running the planning and writing task ##


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Task: 1. Create a bullet-point plan for a summary on the topic: 'The importance of Reinforcement Learning in   │
│  AI'.                                                                                                           │
│  2. Write the summary based on your plan, keeping it around 200 words.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"plan":["Define reinforcement learning and distinguish it from supervised and unsupervised                    │
│  learning","Explain reward-driven, trial-and-error learning and sequential decision-making","Highlight why      │
│  long-term consequences make RL essential for autonomy","List key applications: robotics, games,                │
│  recommendation, control, and LLM alignment","Note major challenges: sample efficiency, stability, reward       │
│  design, and safety","Conclude with RL’s role in adaptive, goal-directed AI systems"],"summary":"Reinforcement  │
│  learning (RL) is essential to AI because it teaches agents to act through trial and error rather than relying  │
│  only on labeled data. By maximizing cumulative reward, RL handles sequential decisions where today’s action    │
│  affects tomorrow’s options—something supervised learning struggles to capture. It complements other AI         │
│  methods by focusing on behavior and consequences, not just patterns. This makes RL central to autonomy:        │
│  robots learn locomotion and manipulation, game agents master complex strategies, recommender systems optimize  │
│  long-term engagement, and control systems adapt in real time. RL also powers modern AI alignment techniques,   │
│  such as reinforcement learning from human feedback, helping large language models become more useful and       │
│  safer. Its emphasis on exploration, feedback, and long-term consequences gives AI systems the ability to       │
│  improve in dynamic, uncertain environments. Still, challenges remain: sample inefficiency, unstable training,  │
│  reward design, and safety. Solving these will determine how widely RL is deployed. As AI moves from            │
│  perception to decision-making, RL’s role will only grow. Overall, RL is not just another AI technique; it is   │
│  the bridge between prediction and purposeful action, making it foundational for adaptive, goal-directed        │
│  intelligent systems."}                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


---
## 校验结果 ##
---
✅ 通过校验，拿到的是: ArticlePlan

计划共 6 条（这是 list，可按下标取）：
  1. Define reinforcement learning and distinguish it from supervised and unsupervised learning
  2. Explain reward-driven, trial-and-error learning and sequential decision-making
  3. Highlight why long-term consequences make RL essential for autonomy
  4. List key applications: robotics, games, recommendation, control, and LLM alignment
  5. Note major challenges: sample efficiency, stability, reward design, and safety
  6. Conclude with RL’s role in adaptive, goal-directed AI systems

正文（字符串，前 200 字符）：
Reinforcement learning (RL) is essential to AI because it teaches agents to act through trial and error rather than relying only on labeled data. By maximizing cumulative reward, RL handles sequential

---
## 原始文本（未加工）##
---
{"plan":["Define reinforcement learning and distinguish it from supervised and unsupervised learning","Explain reward-driven, trial-and-error learning and sequential decision-making","Highlight 